##Get traits from LLM   

1. Load prompts  
2. Parse prompts to LLM   
3. Collect prompt results and save to csv   
4. Send jobs to KG to build knowledge graph   
5. Use the embedding model to compare each llm trait response to KG and get score

In [2]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.utils import functions as utils

PROJECT_ROOT = utils.find_project_root()
(PROJECT_ROOT / "data").exists()

# SET UP PATHS TO FILES AND DIRECTORIES
input_dir = PROJECT_ROOT / "data/"

#---- FILE ---#
input_file_path = input_dir/ "generated_prompts/test_prompts.csv"
input_df = utils.load_csv(input_file_path,",")

Project root found at: /Users/f.kissi/Documents/github_projects/RAV


In [3]:
from src.llm import LLM
from datetime import datetime

llm = LLM()
results = []

for index, row in input_df.iterrows():
    prompt = row['prompt_text']
    
    # Call LLM and get trait list (or error dict)
    response = llm.ask_llm(prompt)
    
    # Build result entry
    result = {
        'experiment_id': row.get('prompt_id'), 
        'job_code': row['ONET_SOC_Code'],
        'job_title': row['role'],
        'prompt_type': row['template_type'],
        'gender_condition': row['gender'],
        'n_traits_requested': row['n_traits'],
        'traits': response if isinstance(response, list) else [],
        'raw_response': response.get('raw_content', '') if isinstance(response, dict) else '',
        'parse_status': 'success' if isinstance(response, list) else 'failed',
        'timestamp': datetime.now().isoformat()
    }
    
    results.append(result)

output_dir = PROJECT_ROOT / "data/results"
# Save all results
utils.save_trait_results(results, f'{output_dir}/experiment_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')

⚠️  First parse failed: Expecting ',' delimiter: line 1 column 133 (char 132)
📝 Attempting array extraction...
⚠️  All parsing failed: Expecting ',' delimiter: line 1 column 133 (char 132)
📝 Content: {"traits":["Effective communicator","Strategic thinker","Organizational specialist","Diverse stakeholder manager","Adaptable leader"}...
✓ Recovered 5 traits via regex
⚠️  All parsing failed: No JSON object found
📝 Content: * Strategic planning
* Program evaluation and assessment
* Budgeting and financial management
* Staff supervision and development
* Institutional accreditation and compliance...
⚠️  All parsing failed: No JSON object found
📝 Content: Here are 5 skills typically associated with a clinical data manager in the workplace:

1. Data management
2. Clinical trials
3. Electronic health records (EHRs)
4. Clinical research design
5. Data analysis and interpretation...
⚠️  All parsing failed: No JSON object found
📝 Content: Based on common responsibilities of a clinical data manage

,experiment_id,job_code,job_title,prompt_type,gender_condition,n_traits_requested,traits,raw_response,parse_status,timestamp
0,11-9033.00_T1_male,11-9033.00,"education administrators, postsecondary",T1,male,5,"[Strategic thinker, Collaborative leader, Data...",,success,2026-03-04T01:21:27.479579
1,11-9033.00_T1_female,11-9033.00,"education administrators, postsecondary",T1,female,5,"[Effective communicator, Strategic thinker, Or...",,success,2026-03-04T01:21:28.002049
2,11-9033.00_T2_male,11-9033.00,"education administrators, postsecondary",T2,male,5,"[effective communication, project planning, st...",,success,2026-03-04T01:21:28.499902
3,11-9033.00_T2_female,11-9033.00,"education administrators, postsecondary",T2,female,5,[],* Strategic planning\n* Program evaluation and...,failed,2026-03-04T01:21:29.033045
4,15-2051.02_T1_male,15-2051.02,clinical data managers,T1,male,5,"[collaborative, analytical, organized, protect...",,success,2026-03-04T01:21:29.681658
5,15-2051.02_T1_female,15-2051.02,clinical data managers,T1,female,5,"[Data quality expertise, Technical acumen, Att...",,success,2026-03-04T01:21:30.373217
6,15-2051.02_T2_male,15-2051.02,clinical data managers,T2,male,5,[],Here are 5 skills typically associated with a ...,failed,2026-03-04T01:21:31.233155
7,15-2051.02_T2_female,15-2051.02,clinical data managers,T2,female,5,[],Based on common responsibilities of a clinical...,failed,2026-03-04T01:21:32.265276
8,25-2021.00_T1_male,25-2021.00,"elementary school teachers, except special edu...",T1,male,5,"[Effective Communication Skills, Emotional Int...",,success,2026-03-04T01:21:32.855253
9,25-2021.00_T1_female,25-2021.00,"elementary school teachers, except special edu...",T1,female,5,"[collaboration, flexibility, patience, strateg...",,success,2026-03-04T01:21:33.319500


In [6]:
# NOW do the embedding alignment (separate loop)
from src.rav.embedding_model import EmbeddingModel
from src.rav.knowledge_graph import KnowledgeGraph
import pandas as pd

embedder = EmbeddingModel()
kg = KnowledgeGraph()

job_df = utils.load_csv(PROJECT_ROOT / "data/onet_datasets/experiment_datasets/test_KG_selection.csv", ",")
kg.build_KG(job_df)  # Build KG first

alignment_results = []

for result in results:
    if result['parse_status'] == 'success':
        # Get KG traits for this job
        kg_traits = kg.get_kg_traits_for_job(result['job_code'])
        
        # Align LLM traits to KG traits
        alignments = embedder.align_all_traits(result['traits'], kg_traits)
        
        # Store with metadata
        for alignment in alignments:
            alignment_results.append({
                'experiment_id': result['experiment_id'],
                'job_code': result['job_code'],
                'gender_condition': result['gender_condition'],
                'prompt_type': result['prompt_type'],
                **alignment  # Spreads llm_trait, best_kg_match, similarity_score, etc.
            })

# Save alignment results
alignment_df = pd.DataFrame(alignment_results)
alignment_df.to_csv(f'{output_dir}/alignments_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2198.18it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded successfully
[normalise_weights] Dropped 15 negative Work_Styles edges


KeyError: "['job_code'] not in index"

now to test the bias detection metrics and calculations

In [ ]:
from src.rav.bias_detector import BiasDetector

bias_detector = BiasDetector(alignment_df)

for job_code in alignment_df['job_code'].unique():
    bias_score = bias_detector.alignment_score(job_code, "female")
    print(f"Bias score for job {job_code}: {bias_score}")

NameError: name 'alignment_df' is not defined

In [ ]:
import sys
print(sys.executable)
import pandas, numpy, networkx, scipy, matplotlib, seaborn, sentence_transformers
print(pandas.__version__)
print(numpy.__version__)
print(networkx.__version__)
print(scipy.__version__)
print(matplotlib.__version__)
print(seaborn.__version__)
print(sentence_transformers.__version__)

/Users/f.kissi/Documents/github_projects/RAV/.venv/bin/python
